# ЛР1. Реалізація та перевірка зворотного поширення похибки

**Виконав:** Швидкий Олександр, КМ-31

Ноутбук — покрокова демонстрація роботи. Уся логіка міститься в `.py`-модулях (`data.py`, `model_numpy.py`, `model_torch.py`, `checks.py`), ноутбук лише імпортує їх і показує проміжні результати. Основне відтворення: `uv run python main.py`; ноутбук: `uv run --group notebook jupyter lab lab1.ipynb`.

In [21]:
import numpy as np
from IPython.display import Markdown, display

from data import load_data
from model_numpy import NumpyMLP, init_params, cross_entropy, PARAM_NAMES
from checks import compare_with_torch, numerical_check
from report import torch_table, numeric_table

np.set_printoptions(precision=5, suppress=True)

## 1. Дані
Стратифікований поділ 35/15 на клас через `default_rng(0)`, стандартизація статистиками навчальної вибірки (`ddof=0`).

In [22]:
ds = load_data(seed=0)
X, y = ds.X_train, ds.y_train
print("train:", X.shape, "класи:", np.bincount(y))
print("test: ", ds.X_test.shape, "класи:", np.bincount(ds.y_test))
print("mean =", ds.mean)
print("std  =", ds.std)
print("після стандартизації: mean ≈", X.mean(0).round(12), " std =", X.std(0))

train: (105, 4) класи: [35 35 35]
test:  (45, 4) класи: [15 15 15]
mean = [5.82286 3.05905 3.78095 1.20667]
std  = [0.83258 0.41672 1.7691  0.75584]
після стандартизації: mean ≈ [-0. -0.  0.  0.]  std = [1. 1. 1. 1.]


## 2. Ініціалізація
Новий `default_rng(0)`: спочатку $W_1 \sim \mathcal N(0, \sqrt{2/4})$ (He), потім $W_2 \sim \mathcal N(0, \sqrt{2/11})$ (Xavier); зсуви — нулі.

In [23]:
params = init_params(seed=0)
for k, v in params.items():
    print(k, v.shape, v.dtype)
print("\nW1 =\n", params["W1"])

W1 (4, 8) float64
b1 (8,) float64
W2 (8, 3) float64
b2 (3,) float64

W1 =
 [[ 0.0889  -0.09341  0.45285  0.07418 -0.37878  0.25569  0.92207  0.66969]
 [-0.49762 -0.89479 -0.44072  0.02922 -1.64405 -0.15471 -0.88099 -0.51779]
 [-0.38485 -0.22366  0.29107  0.73717 -0.09089  0.96624 -0.47036  0.24856]
 [ 0.63885  0.06648 -0.52573 -0.65176 -0.32366  0.1557  -0.71391 -0.14791]]


## 3. Прямий прохід і стабільна крос-ентропія
$Z_1 = XW_1 + b_1,\ H = \max(Z_1,0),\ Z_2 = HW_2 + b_2$; log-softmax зі зсувом на максимум рядка.

In [24]:
model = NumpyMLP(params)
loss = model.forward(X, y)
print(f"L = {loss:.16f}   (ln 3 = {np.log(3):.4f})")
print("\nЗбережені проміжні значення:")
for k, v in model.cache.items():
    print(f"  {k:10s} {v.shape}")

L = 1.4562007801142816   (ln 3 = 1.0986)

Збережені проміжні значення:
  X          (105, 4)
  relu_mask  (105, 8)
  H          (105, 8)
  P          (105, 3)
  y          (105,)


In [25]:
# Стабільність: великі логіти
Z = np.array([[1000.0, 0.0, -1000.0], [-1000.0, -1001.0, -1002.0]])
t = np.array([1, 0])
with np.errstate(all="ignore"):
    naive = -np.log(np.exp(Z) / np.exp(Z).sum(1, keepdims=True))[np.arange(2), t].mean()
print("наївна CE:  ", naive)
print("стабільна CE:", cross_entropy(Z, t)[0])

наївна CE:   nan
стабільна CE: 500.2038029822222


## 4. Ручний зворотний прохід
$$\frac{\partial L}{\partial Z_2} = \frac{P-Y}{N},\quad
\frac{\partial L}{\partial W_2} = H^\top \frac{\partial L}{\partial Z_2},\quad
\frac{\partial L}{\partial b_2} = \sum_n \frac{\partial L}{\partial Z_2},\quad
\frac{\partial L}{\partial Z_1} = \frac{\partial L}{\partial Z_2} W_2^\top \odot [Z_1>0],\quad
\frac{\partial L}{\partial W_1} = X^\top \frac{\partial L}{\partial Z_1},\quad
\frac{\partial L}{\partial b_1} = \sum_n \frac{\partial L}{\partial Z_1}$$

In [26]:
grads = model.backward()
for k in PARAM_NAMES:
    print(f"d{k}: {grads[k].shape}  (параметр {params[k].shape})")
print("\nСума елементів db2 (має бути ≈ 0):", grads["b2"].sum())

dW1: (4, 8)  (параметр (4, 8))
db1: (8,)  (параметр (8,))
dW2: (8, 3)  (параметр (8, 3))
db2: (3,)  (параметр (3,))

Сума елементів db2 (має бути ≈ 0): 1.0408340855860843e-17


## 5. Звірка з PyTorch (поріг $10^{-12}$)

In [27]:
res = compare_with_torch(params, X, y)
display(Markdown(torch_table(res)))

| Величина | Максимальна абсолютна різниця NumPy / PyTorch | Перевірку пройдено |
|---|---|---|
| Втрата | 2.220e-16 | так |
| Градієнт W1 | 2.776e-17 | так |
| Градієнт b1 | 5.204e-17 | так |
| Градієнт W2 | 5.551e-17 | так |
| Градієнт b2 | 1.249e-16 | так |

Втрата NumPy:   `1.4562007801142816`  
Втрата PyTorch: `1.4562007801142818`

## 6. Чисельна перевірка ($\varepsilon = 10^{-6}$, поріг $10^{-7}$)

In [28]:
num = numerical_check(params, X, y)
display(Markdown(numeric_table(num)))
Z1 = X @ params["W1"] + params["b1"]
print("min |Z1| =", np.abs(Z1).min(), " — далеко від зламу ReLU порівняно з ε·max|x| ≈", 1e-6 * np.abs(X).max())

| Параметр | Градієнт backward() | Чисельна похідна | Абсолютна різниця | Перевірку пройдено |
|---|---|---|---|---|
| W1[0, 0] | -1.992018433056e-02 | -1.992018439090e-02 | 6.034e-11 | так |
| b1[0] | -7.056611158438e-02 | -7.056611162071e-02 | 3.633e-11 | так |
| W2[0, 0] | 1.352302086055e-01 | 1.352302086977e-01 | 9.218e-11 | так |
| b2[0] | 8.552164484196e-02 | 8.552164498798e-02 | 1.460e-10 | так |

min |Z1| = 0.0006109391964872219  — далеко від зламу ReLU порівняно з ε·max|x| ≈ 2.541384030978644e-06


## 7. Дослід із навмисною помилкою
Прибираємо ділення на $N$ у $\partial L/\partial Z_2$ (`divide_by_n=False`); втрата, еталон PyTorch і формула чисельної похідної не змінюються.

**Прогноз:** втрата не зміниться; усі ненульові градієнти зростуть рівно в $N=105$ разів; звірка з PyTorch провалиться для всіх чотирьох градієнтів (втрата пройде), чисельна перевірка — для всіх чотирьох параметрів.

In [29]:
loss_bug, g_bug = NumpyMLP(params).loss_and_grads(X, y, divide_by_n=False)
print("втрата: правильна =", loss, " з помилкою =", loss_bug, " різниця =", abs(loss - loss_bug))
for k in PARAM_NAMES:
    r = g_bug[k] / grads[k]
    print(f"{k}: відношення в межах [{r.min():.10f}, {r.max():.10f}]")

втрата: правильна = 1.4562007801142816  з помилкою = 1.4562007801142816  різниця = 0.0
W1: відношення в межах [105.0000000000, 105.0000000000]
b1: відношення в межах [105.0000000000, 105.0000000000]
W2: відношення в межах [105.0000000000, 105.0000000000]
b2: відношення в межах [105.0000000000, 105.0000000000]


In [30]:
display(Markdown("**Звірка з PyTorch (з помилкою)**\n\n" + torch_table(compare_with_torch(params, X, y, divide_by_n=False))))
display(Markdown("**Чисельна перевірка (з помилкою)**\n\n" + numeric_table(numerical_check(params, X, y, divide_by_n=False))))

**Звірка з PyTorch (з помилкою)**

| Величина | Максимальна абсолютна різниця NumPy / PyTorch | Перевірку пройдено |
|---|---|---|
| Втрата | 2.220e-16 | так |
| Градієнт W1 | 1.852e+01 | **ні** |
| Градієнт b1 | 1.564e+01 | **ні** |
| Градієнт W2 | 3.419e+01 | **ні** |
| Градієнт b2 | 1.120e+01 | **ні** |

Втрата NumPy:   `1.4562007801142816`  
Втрата PyTorch: `1.4562007801142818`

**Чисельна перевірка (з помилкою)**

| Параметр | Градієнт backward() | Чисельна похідна | Абсолютна різниця | Перевірку пройдено |
|---|---|---|---|---|
| W1[0, 0] | -2.091619354708e+00 | -1.992018439090e-02 | 2.072e+00 | **ні** |
| b1[0] | -7.409441716360e+00 | -7.056611162071e-02 | 7.339e+00 | **ні** |
| W2[0, 0] | 1.419917190357e+01 | 1.352302086977e-01 | 1.406e+01 | **ні** |
| b2[0] | 8.979772708405e+00 | 8.552164498798e-02 | 8.894e+00 | **ні** |

**Висновок досліду:** прогноз підтвердився повністю — втрата збігається біт-у-біт, градієнти масштабовані рівно в 105 разів, обидві перевірки виявили помилку. Правильна реалізація лишається за замовчуванням (`divide_by_n=True`).